In [11]:
#from Bio.PDB import *
#from Bio.PDB.DSSP import DSSP
#from Bio.PDB.DSSP import dssp_dict_from_pdb_file
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.lib.formats.libdcd import DCDFile

#dcd_file='/media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-{}/analyses/traj{}/scripts/trajectory.dcd'
#faire 3 batch (1 de 400 à 600, 2 de 600 à 800, 3 de 800 à 1000)
dcd_file='/media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-{}/analyses/traj{}/data/batch{}.dcd'
psf_file='/media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-{}/analyses/traj{}/data/analog-{}.psf'

coor_file='/media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-{}/analyses/traj{}/scripts/ring_coor/{}.dat'

In [12]:
# Ajouter cette fonction au début du notebook

def is_monomer(u, resid, resname, cutoff=6.0):
    """
    Vérifie si un résidu est un monomère (pas d'autre analogue à moins de cutoff Å).
    Ne considère que les atomes lourds (pas H).
    
    Parameters:
    -----------
    u : MDAnalysis.Universe
    resid : int - identifiant du résidu à tester
    resname : str - nom du résidu (SCF, SCY, SCW, etc.)
    cutoff : float - distance seuil en Å
    
    Returns:
    --------
    bool : True si monomère, False sinon
    """
    # Atomes lourds du résidu courant
    current_res = u.select_atoms(f'resname {resname} and resid {resid} and not name H*')
    
    # Atomes lourds de tous les autres analogues
    other_analogs = u.select_atoms(f'resname {resname} and not resid {resid} and not name H*')
    
    if len(other_analogs) == 0:
        return True  # Seul analogue = monomère par définition
    
    # Calcul des distances minimales
    from MDAnalysis.lib.distances import distance_array
    distances = distance_array(current_res.positions, other_analogs.positions)
    
    # Si la distance minimale est > cutoff, c'est un monomère
    return distances.min() > cutoff


def get_monomer_resids(u, resname, cutoff=6.0):
    """
    Retourne la liste des resid qui sont des monomères pour une frame donnée.
    """
    all_resids = u.select_atoms(f'resname {resname}').resids
    unique_resids = np.unique(all_resids)
    
    monomer_resids = []
    for resid in unique_resids:
        if is_monomer(u, resid, resname, cutoff):
            monomer_resids.append(resid)
    
    return monomer_resids

In [13]:
#get coordinates
def get_coord(structure):
    # Create an empty list to store atom coordinates
    atom_coordinates = []

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    atom_coordinates.append(atom.get_coord())
    return atom_coordinates

def get_ca_coord(structure):
    # Create an empty list to store atom coordinates
    calpha_atoms = []

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                   if atom.get_name()=="CA":
                       calpha_atoms.append(atom) 
    CA_coordinates = [atom.get_coord() for atom in calpha_atoms]
    return CA_coordinates

#save new coordinates
def update(structure, modified_coordinates):
    # Update the atom coordinates in the structure:
    atom_index = 0

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    # Update the atom coordinates in the structure
                    atom.set_coord(modified_coordinates[atom_index])
                    atom_index += 1
    return structure

In [14]:
def nglplot(x):
    traj = pt.load('./'+x)
    view = nv.show_pytraj(traj)



In [15]:
def regressi_vector(x, y, z):
    #determine a vector of the regression
    x1=np.column_stack((x,z))
    model = linear_model.LinearRegression()
    #print(x1, y)
    model.fit(x1,y)
    v1 = [model.coef_[0], 1, model.coef_[1]]
    return v1 / np.linalg.norm(v1) #vecteur unitaire
    #vecteur_directeur = np.array([reg.coef_[0], reg.coef_[1], reg.coef_[2]])

def angle_between(v1, v2):
    #v1 et v2 doivent être des vecteurs unitaires
    return np.arccos(np.clip(np.dot(v1, v2), -1.0, 1.0))


def angle_with_normal(structure, TM, v2):
    #calculate angle between the vector and the normal of the membrane
    x, y, z = get_ca_coord_part(structure, TM)
    v1=regressi_vector(x, y, z)
    return angle_between(v1, v2)

In [16]:
pdb_parser = MMCIFParser(QUIET=True)
structure1 = pdb_parser.get_structure("IgaA", "./protWT_10.cif")

NameError: name 'MMCIFParser' is not defined

In [17]:
print(dcd_file.format('SCF', 1))
with DCDFile(dcd_file.format('SCF', 1)) as f:
    for frame in f:
        print(frame.atoms.select_atoms('resname SCW'))

IndexError: Replacement index 2 out of range for positional args tuple

In [19]:
# Version optimisée : extraction de tous les carbones en UN SEUL parcours de trajectoire
#scs = ['SCW', 'SCF', 'SCY', 'SCYM']
scs = ['SCYM']
cutoff = 6.0

for sc in scs:
    if sc == 'SCW':
        trajs = [4, 2, 3]
        carbons = ['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
    elif sc == 'SCYM':
        trajs = [4, 5] #[1, 2, 6]
        carbons = ['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
    else:
        trajs = [1, 2, 3]
        carbons = ['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
    
    for traj_idx in trajs:
        for k in range(1, 4):
            u = mda.Universe(psf_file.format(sc, traj_idx, sc), dcd_file.format(sc, traj_idx, k))
            print(f'\n=== {sc} traj{traj_idx} batch{k} ===')
            print(u)
        
            # Initialiser une liste par carbone (plus rapide que df.loc)
            data = {c: [] for c in carbons}
        
            # Sélection MDAnalysis pour TOUS les carbones du cycle
            all_carbons_str = " ".join(carbons)
        
            # UN SEUL PARCOURS de la trajectoire
            for ts in u.trajectory:
                fr = ts.frame
                if fr % 1000 == 0:
                    print(f'Frame {fr//1000}/20')
            
                # Obtenir les monomères UNE SEULE FOIS par frame
                monomer_resids = get_monomer_resids(u, sc, cutoff)
            
                if len(monomer_resids) == 0:
                    continue
            
                resid_selection = " ".join(map(str, monomer_resids))
            
                # Sélectionner TOUS les atomes du cycle en une seule requête
                ring_atoms = u.select_atoms(f'resname {sc} and name {all_carbons_str} and resid {resid_selection}')
            
                # Trier les données par type de carbone
                for atom in ring_atoms:
                    c = atom.name
                    data[c].append([fr, c, atom.index, atom.resid, *atom.position])
        
            # Sauvegarder chaque carbone dans un fichier séparé
            for c in carbons:
                df = pd.DataFrame(data[c], columns=['#frame', 'C_type', 'index', 'resid', 'x', 'y', 'z'])
                output_file = coor_file.format(sc, traj_idx, f'{c}_coor_mono_{k}')
                np.savetxt(
                    output_file,
                    df,
                    fmt='%i %s %i %i %.6f %.6f %.6f',
                    delimiter=' ',
                    newline='\n',
                    header='frame C_type index resid x y z',
                    comments='#'
                )
                print(f'  Saved {c}: {len(df)} entries (batch {k})')

print("\nC'est bon, j'ai fini !!!")



=== SCYM traj1 batch1 ===
<Universe with 390 atoms>
Frame 0/20
Frame 1/20
  Saved CG: 43173 entries (batch 1)
  Saved CD1: 43173 entries (batch 1)
  Saved CD2: 43173 entries (batch 1)
  Saved CE1: 43173 entries (batch 1)
  Saved CE2: 43173 entries (batch 1)
  Saved CZ: 43173 entries (batch 1)

=== SCYM traj1 batch2 ===
<Universe with 390 atoms>
Frame 0/20
Frame 1/20
  Saved CG: 43175 entries (batch 2)
  Saved CD1: 43175 entries (batch 2)
  Saved CD2: 43175 entries (batch 2)
  Saved CE1: 43175 entries (batch 2)
  Saved CE2: 43175 entries (batch 2)
  Saved CZ: 43175 entries (batch 2)

=== SCYM traj1 batch3 ===
<Universe with 390 atoms>
Frame 0/20
Frame 1/20
Frame 2/20
  Saved CG: 43336 entries (batch 3)
  Saved CD1: 43336 entries (batch 3)
  Saved CD2: 43336 entries (batch 3)
  Saved CE1: 43336 entries (batch 3)
  Saved CE2: 43336 entries (batch 3)
  Saved CZ: 43336 entries (batch 3)

=== SCYM traj2 batch1 ===
<Universe with 390 atoms>
Frame 0/20
Frame 1/20
  Saved CG: 42812 entries (ba

In [9]:
#carbons=['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
#carbons=['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
carbons=['CE1', 'CD2', 'CG', 'ND1', 'NE2']
sc='SCHD'
t=1
for k in range(2,6):
    u=mda.Universe(psf_file.format(sc, t), dcd_file.format(sc, t, k))
    print(u)
    print('Working on bloc{}.dcd ...'.format(k))
    #cg=u.select_atoms('resname SCF and name CG')
    #print(len(cg))
    for j in range(len(carbons)):
        df=pd.DataFrame(columns=['#frame', 'C_type', 'index', 'x', 'y', 'z'])
        c=carbons[j]
        print('Working on carbon {} ...'.format(c))
        for ts in u.trajectory:
            #print(u.trajectory.frame)
            fr=u.trajectory.frame
            if fr%1000==0:
                print('Actual step : ', fr/1000, ' over 20' )
            ca=u.select_atoms('resname {} and name {}'.format(sc, c))
            for i in range(len(ca)):
                x, y, z = ca.positions[i][0], ca.positions[i][1], ca.positions[i][2]
                df.loc[len(df.index)] = [fr, c, i, x, y, z]
        np.savetxt(coor_file.format(sc, t, '{}_coor_'.format(c), k), df, fmt='%i %s %i %.6f %.6f %.6f', delimiter=' ', newline='\n', header='frame C_type index x y z', comments='#')

print('C\'est bon, j\'ai fini !!!')

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc2.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc3.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc4.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc5.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

In [10]:
#carbons=['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
#carbons=['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
carbons=['CE1', 'CD2', 'CG', 'ND1', 'NE2']
sc='SCHD'
t=2
for k in range(2,6):
    u=mda.Universe(psf_file.format(sc, t), dcd_file.format(sc, t, k))
    print(u)
    print('Working on bloc{}.dcd ...'.format(k))
    #cg=u.select_atoms('resname SCF and name CG')
    #print(len(cg))
    for j in range(len(carbons)):
        df=pd.DataFrame(columns=['#frame', 'C_type', 'index', 'x', 'y', 'z'])
        c=carbons[j]
        print('Working on carbon {} ...'.format(c))
        for ts in u.trajectory:
            #print(u.trajectory.frame)
            fr=u.trajectory.frame
            if fr%1000==0:
                print('Actual step : ', fr/1000, ' over 20' )
            ca=u.select_atoms('resname {} and name {}'.format(sc, c))
            for i in range(len(ca)):
                x, y, z = ca.positions[i][0], ca.positions[i][1], ca.positions[i][2]
                df.loc[len(df.index)] = [fr, c, i, x, y, z]
        np.savetxt(coor_file.format(sc, t, '{}_coor_'.format(c), k), df, fmt='%i %s %i %.6f %.6f %.6f', delimiter=' ', newline='\n', header='frame C_type index x y z', comments='#')

print('C\'est bon, j\'ai fini !!!')

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc2.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc3.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc4.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc5.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

In [11]:
#carbons=['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
#carbons=['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
carbons=['CE1', 'CD2', 'CG', 'ND1', 'NE2']
sc='SCHD'
t=3
for k in range(2,6):
    u=mda.Universe(psf_file.format(sc, t), dcd_file.format(sc, t, k))
    print(u)
    print('Working on bloc{}.dcd ...'.format(k))
    #cg=u.select_atoms('resname SCF and name CG')
    #print(len(cg))
    for j in range(len(carbons)):
        df=pd.DataFrame(columns=['#frame', 'C_type', 'index', 'x', 'y', 'z'])
        c=carbons[j]
        print('Working on carbon {} ...'.format(c))
        for ts in u.trajectory:
            #print(u.trajectory.frame)
            fr=u.trajectory.frame
            if fr%1000==0:
                print('Actual step : ', fr/1000, ' over 20' )
            ca=u.select_atoms('resname {} and name {}'.format(sc, c))
            for i in range(len(ca)):
                x, y, z = ca.positions[i][0], ca.positions[i][1], ca.positions[i][2]
                df.loc[len(df.index)] = [fr, c, i, x, y, z]
        np.savetxt(coor_file.format(sc, t, '{}_coor_'.format(c), k), df, fmt='%i %s %i %.6f %.6f %.6f', delimiter=' ', newline='\n', header='frame C_type index x y z', comments='#')

print('C\'est bon, j\'ai fini !!!')

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc2.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc3.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc4.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29033 atoms>
Working on bloc5.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

In [7]:
#carbons=['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
#carbons=['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
carbons=['CE1', 'CD2', 'CG', 'ND1', 'NE2']
sc='SCHE'
t=2
for k in range(2,6):
    u=mda.Universe(psf_file.format(sc, t), dcd_file.format(sc, t, k))
    print(u)
    print('Working on bloc{}.dcd ...'.format(k))
    #cg=u.select_atoms('resname SCF and name CG')
    #print(len(cg))
    for j in range(len(carbons)):
        df=pd.DataFrame(columns=['#frame', 'C_type', 'index', 'x', 'y', 'z'])
        c=carbons[j]
        print('Working on carbon {} ...'.format(c))
        for ts in u.trajectory:
            #print(u.trajectory.frame)
            fr=u.trajectory.frame
            if fr%1000==0:
                print('Actual step : ', fr/1000, ' over 20' )
            ca=u.select_atoms('resname {} and name {}'.format(sc, c))
            for i in range(len(ca)):
                x, y, z = ca.positions[i][0], ca.positions[i][1], ca.positions[i][2]
                df.loc[len(df.index)] = [fr, c, i, x, y, z]
        np.savetxt(coor_file.format(sc, t, '{}_coor_'.format(c), k), df, fmt='%i %s %i %.6f %.6f %.6f', delimiter=' ', newline='\n', header='frame C_type index x y z', comments='#')

print('C\'est bon, j\'ai fini !!!')

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29078 atoms>
Working on bloc2.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29078 atoms>
Working on bloc3.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29078 atoms>
Working on bloc4.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29078 atoms>
Working on bloc5.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

In [8]:
#carbons=['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
#carbons=['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
carbons=['CE1', 'CD2', 'CG', 'ND1', 'NE2']
sc='SCHE'
t=3
for k in range(2,6):
    u=mda.Universe(psf_file.format(sc, t), dcd_file.format(sc, t, k))
    print(u)
    print('Working on bloc{}.dcd ...'.format(k))
    #cg=u.select_atoms('resname SCF and name CG')
    #print(len(cg))
    for j in range(len(carbons)):
        df=pd.DataFrame(columns=['#frame', 'C_type', 'index', 'x', 'y', 'z'])
        c=carbons[j]
        print('Working on carbon {} ...'.format(c))
        for ts in u.trajectory:
            #print(u.trajectory.frame)
            fr=u.trajectory.frame
            if fr%1000==0:
                print('Actual step : ', fr/1000, ' over 20' )
            ca=u.select_atoms('resname {} and name {}'.format(sc, c))
            for i in range(len(ca)):
                x, y, z = ca.positions[i][0], ca.positions[i][1], ca.positions[i][2]
                df.loc[len(df.index)] = [fr, c, i, x, y, z]
        np.savetxt(coor_file.format(sc, t, '{}_coor_'.format(c), k), df, fmt='%i %s %i %.6f %.6f %.6f', delimiter=' ', newline='\n', header='frame C_type index x y z', comments='#')

print('C\'est bon, j\'ai fini !!!')

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29078 atoms>
Working on bloc2.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29078 atoms>
Working on bloc3.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29078 atoms>
Working on bloc4.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 29078 atoms>
Working on bloc5.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

In [5]:
#carbons=['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
#carbons=['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
carbons=['CE1', 'CD2', 'CG', 'ND1', 'NE2']
sc='SCHP'
t=3
for k in range(2,6):
    u=mda.Universe(psf_file.format(sc+'-N', t), dcd_file.format(sc+'-N', t, k))
    print(u)
    print('Working on bloc{}.dcd ...'.format(k))
    #cg=u.select_atoms('resname SCF and name CG')
    #print(len(cg))
    for j in range(len(carbons)):
        df=pd.DataFrame(columns=['#frame', 'C_type', 'index', 'x', 'y', 'z'])
        c=carbons[j]
        print('Working on carbon {} ...'.format(c))
        for ts in u.trajectory:
            #print(u.trajectory.frame)
            fr=u.trajectory.frame
            if fr%1000==0:
                print('Actual step : ', fr/1000, ' over 20' )
            ca=u.select_atoms('resname {} and name {}'.format(sc, c))
            for i in range(len(ca)):
                x, y, z = ca.positions[i][0], ca.positions[i][1], ca.positions[i][2]
                df.loc[len(df.index)] = [fr, c, i, x, y, z]
        np.savetxt(coor_file.format(sc+'-N', t, '{}_coor_'.format(c), k), df, fmt='%i %s %i %.6f %.6f %.6f', delimiter=' ', newline='\n', header='frame C_type index x y z', comments='#')

print('C\'est bon, j\'ai fini !!!')

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28848 atoms>
Working on bloc2.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28848 atoms>
Working on bloc3.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28848 atoms>
Working on bloc4.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28848 atoms>
Working on bloc5.dcd ...
Working on carbon CE1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

In [6]:
#carbons=['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
carbons=['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
sc='SCW'
t=2
for k in range(2,6):
    u=mda.Universe(psf_file.format(sc, t), dcd_file.format(sc, t, k))
    print(u)
    print('Working on bloc{}.dcd ...'.format(k))
    #cg=u.select_atoms('resname SCF and name CG')
    #print(len(cg))
    for j in range(len(carbons)):
        df=pd.DataFrame(columns=['#frame', 'C_type', 'index', 'x', 'y', 'z'])
        c=carbons[j]
        print('Working on carbon {} ...'.format(c))
        for ts in u.trajectory:
            #print(u.trajectory.frame)
            fr=u.trajectory.frame
            if fr%1000==0:
                print('Actual step : ', fr/1000, ' over 20' )
            ca=u.select_atoms('resname {} and name {}'.format(sc, c))
            for i in range(len(ca)):
                x, y, z = ca.positions[i][0], ca.positions[i][1], ca.positions[i][2]
                df.loc[len(df.index)] = [fr, c, i, x, y, z]
        np.savetxt(coor_file.format(sc, t, '{}_coor_'.format(c), k), df, fmt='%i %s %i %.6f %.6f %.6f', delimiter=' ', newline='\n', header='frame C_type index x y z', comments='#')

print('C\'est bon, j\'ai fini !!!')

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28798 atoms>
Working on bloc2.dcd ...
Working on carbon CD1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28798 atoms>
Working on bloc3.dcd ...
Working on carbon CD1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28798 atoms>
Working on bloc4.dcd ...
Working on carbon CD1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28798 atoms>
Working on bloc5.dcd ...
Working on carbon CD1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

In [7]:
carbons=['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
#carbons=['CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ']
sc='SCW'
t=3
for k in range(2,6):
    u=mda.Universe(psf_file.format(sc, t), dcd_file.format(sc, t, k))
    print(u)
    print('Working on bloc{}.dcd ...'.format(k))
    #cg=u.select_atoms('resname SCF and name CG')
    #print(len(cg))
    for j in range(len(carbons)):
        df=pd.DataFrame(columns=['#frame', 'C_type', 'index', 'x', 'y', 'z'])
        c=carbons[j]
        print('Working on carbon {} ...'.format(c))
        for ts in u.trajectory:
            #print(u.trajectory.frame)
            fr=u.trajectory.frame
            if fr%1000==0:
                print('Actual step : ', fr/1000, ' over 20' )
            ca=u.select_atoms('resname {} and name {}'.format(sc, c))
            for i in range(len(ca)):
                x, y, z = ca.positions[i][0], ca.positions[i][1], ca.positions[i][2]
                df.loc[len(df.index)] = [fr, c, i, x, y, z]
        np.savetxt(coor_file.format(sc, t, '{}_coor_'.format(c), k), df, fmt='%i %s %i %.6f %.6f %.6f', delimiter=' ', newline='\n', header='frame C_type index x y z', comments='#')

print('C\'est bon, j\'ai fini !!!')

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28798 atoms>
Working on bloc2.dcd ...
Working on carbon CD1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28798 atoms>
Working on bloc3.dcd ...
Working on carbon CD1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28798 atoms>
Working on bloc4.dcd ...
Working on carbon CD1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


<Universe with 28798 atoms>
Working on bloc5.dcd ...
Working on carbon CD1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  1

In [9]:
carbons=['CD1', 'CD2', 'CE2', 'CZ2', 'CH2', 'CZ3', 'CE3']
sc='SCW'
t=4
for k in range(5,6):
    u=mda.Universe(psf_file.format(sc, t), dcd_file.format(sc, t, k))
    #print(u)
    print('Working on bloc{}.dcd ...'.format(k))
    #cg=u.select_atoms('resname SCF and name CG')
    #print(len(cg))
    for j in range(len(carbons)):
        df=pd.DataFrame(columns=['#frame', 'C_type', 'index', 'x', 'y', 'z'])
        c=carbons[j]
        print('Working on carbon {} ...'.format(c))
        for ts in u.trajectory:
            #print(u.trajectory.frame)
            fr=u.trajectory.frame
            if fr%1000==0:
                print('Actual step : ', int(fr/1000), ' over 20' )
            ca=u.select_atoms('resname {} and name {}'.format(sc, c))
            for i in range(len(ca)):
                x, y, z = ca.positions[i][0], ca.positions[i][1], ca.positions[i][2]
                df.loc[len(df.index)] = [fr, c, i, x, y, z]
        np.savetxt(coor_file.format(sc, t, '{}_coor_'.format(c), k), df, fmt='%i %s %i %.6f %.6f %.6f', delimiter=' ', newline='\n', header='frame C_type index x y z', comments='#')

print('C\'est bon, j\'ai fini !!!')

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


Working on bloc5.dcd ...
Working on carbon CD1 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  12.0  over 20
Actual step :  13.0  over 20
Actual step :  14.0  over 20
Actual step :  15.0  over 20
Actual step :  16.0  over 20
Actual step :  17.0  over 20
Actual step :  18.0  over 20
Actual step :  19.0  over 20
Working on carbon CD2 ...
Actual step :  0.0  over 20
Actual step :  1.0  over 20
Actual step :  2.0  over 20
Actual step :  3.0  over 20
Actual step :  4.0  over 20
Actual step :  5.0  over 20
Actual step :  6.0  over 20
Actual step :  7.0  over 20
Actual step :  8.0  over 20
Actual step :  9.0  over 20
Actual step :  10.0  over 20
Actual step :  11.0  over 20
Actual step :  